In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score
import plotly.express as px
import geopandas as gpd
import os

# ==============================================================================
# PASSO 1: LIMPEZA E PREPARAÇÃO DOS DADOS (VERSÃO FINAL COM DENSIDADE)
# ==============================================================================
print(">>> [FASE 1] Carregando e limpando os arquivos do IBGE...")

NOME_DA_PASTA = 'dados_brutos'
caminho_pib = os.path.join(NOME_DA_PASTA, 'pib.xlsx')
caminho_alfabetizacao = os.path.join(NOME_DA_PASTA, 'alfabetizacao.xlsx')
caminho_densidade = os.path.join(NOME_DA_PASTA, 'densidade.xlsx') # NOVO ARQUIVO

# --- 1. Limpeza do PIB ---
df_pib = pd.read_excel(caminho_pib, skiprows=4, header=None, engine='openpyxl')
df_pib = df_pib.iloc[1:, [0, -1]]
df_pib.columns = ['codigo_ibge', 'pib_per_capita']
df_pib = df_pib.dropna()
df_pib['codigo_ibge'] = pd.to_numeric(df_pib['codigo_ibge'], errors='coerce')

# --- 2. Limpeza da Alfabetização ---
df_alfabetizacao = pd.read_excel(caminho_alfabetizacao, skiprows=4, header=None, engine='openpyxl')
df_alfabetizacao = df_alfabetizacao.iloc[1:, [0, -1]]
df_alfabetizacao.columns = ['codigo_ibge', 'taxa_alfabetizacao']
df_alfabetizacao = df_alfabetizacao.dropna()
df_alfabetizacao['codigo_ibge'] = pd.to_numeric(df_alfabetizacao['codigo_ibge'], errors='coerce')
df_alfabetizacao['taxa_alfabetizacao'] = pd.to_numeric(df_alfabetizacao['taxa_alfabetizacao'], errors='coerce')

# --- 3. Limpeza da Densidade ---
df_densidade = pd.read_excel(caminho_densidade, skiprows=4, header=None, engine='openpyxl')
df_densidade = df_densidade.iloc[1:, [0, -1]]
df_densidade.columns = ['codigo_ibge', 'densidade_demografica']
df_densidade = df_densidade.dropna()
df_densidade['codigo_ibge'] = pd.to_numeric(df_densidade['codigo_ibge'], errors='coerce')

# --- UNIFICAÇÃO FINAL ---
print(">>> Unificando os dados...")

# Limpeza e conversão final
df_pib = df_pib.dropna().astype({'codigo_ibge': 'int64'})
df_alfabetizacao = df_alfabetizacao.dropna().astype({'codigo_ibge': 'int64'})
df_densidade = df_densidade.dropna().astype({'codigo_ibge': 'int64'})

# Junção
df_final = pd.merge(df_pib, df_alfabetizacao, on='codigo_ibge', how='inner')
df_final = pd.merge(df_final, df_densidade, on='codigo_ibge', how='inner')

print(f"\n>>> SUCESSO! DADOS CONSOLIDADOS! Total de {len(df_final)} municípios com dados completos.")
print(df_final.head())
df_final.info()

In [ ]:
# ==============================================================================
# FASES 2 A 6: ANÁLISE E VISUALIZAÇÃO PARA APRESENTAÇÃO
# ==============================================================================

# PASSO 2: NORMALIZAÇÃO (sem alterações)
print("\n>>> [FASE 2] Normalizando os dados...")
indicadores = ['pib_per_capita', 'taxa_alfabetizacao', 'densidade_demografica']
scaler = StandardScaler()
df_normalizado = pd.DataFrame(scaler.fit_transform(df_final[indicadores]), columns=indicadores)

In [ ]:
# PASSO 3: DETERMINAÇÃO DO K (sem alterações, apenas mostrando os gráficos)
print("\n>>> [FASE 3] Mostrando os gráficos para a escolha do K ideal...")
# (Os cálculos já foram feitos, aqui apenas re-geramos os gráficos para visualização)
inercias, silhuetas = [], []
k_range = range(2, 11)
for k in k_range:
    modelo = KMedoids(n_clusters=k, random_state=42)
    labels = modelo.fit_predict(df_normalizado)
    inercias.append(modelo.inertia_)
    silhuetas.append(silhouette_score(df_normalizado, labels))

px.line(x=k_range, y=inercias, title='<b>Método do Cotovelo</b>', markers=True).show()
px.bar(x=k_range, y=silhuetas, title='<b>Análise da Silhueta</b>').show()
K_ESCOLHIDO = 4 
print(f">>> K escolhido = {K_ESCOLHIDO} perfis.")

In [ ]:
# PASSO 4: ALGORITMO K-MEDOID (sem alterações)
print(f"\n>>> [FASE 4] Segmentando os municípios...")
modelo_final = KMedoids(n_clusters=K_ESCOLHIDO, random_state=42)
clusters = modelo_final.fit_predict(df_normalizado)
df_final['cluster_num'] = clusters

# ==============================================================================
# MELHORIA VISUAL 1: NOMEAR E ORDENAR OS CLUSTERS
# ==============================================================================
perfil_para_ordenar = df_final.groupby('cluster_num')[indicadores].mean().round(2)

# Ordena os clusters com base no PIB per capita (do menor para o maior)
perfil_ordenado = perfil_para_ordenar.sort_values('pib_per_capita')

# Cria os nomes para cada grupo
nomes_dos_perfis = [
    "Municípios Vulneráveis", 
    "Cidades Médias",
    "Potências do Agronegócio",
    "Polos Urbanos Desenvolvidos"
]
perfil_ordenado['Perfil'] = nomes_dos_perfis

# Cria um "dicionário" para substituir os números (0,1,2,3) pelos nomes
mapa_nomes = perfil_ordenado['Perfil'].to_dict()
print("\nNomes definidos para cada cluster:")
print(mapa_nomes)
# Aplica os nomes à nossa tabela final
df_final['perfil'] = df_final['cluster_num'].map(mapa_nomes)

In [ ]:
# PASSO 5: VISUALIZAÇÃO
print("\n>>> [FASE 5] Gerando gráficos para a apresentação...")

# --- GRÁFICO DE RADAR COM NOMES ---
df_normalizado['cluster_num'] = clusters
df_normalizado['perfil'] = df_normalizado['cluster_num'].map(mapa_nomes)
df_radar = df_normalizado.groupby('perfil')[indicadores].mean().reset_index()
df_radar_melted = pd.melt(df_radar, id_vars=['perfil'], var_name='Indicador', value_name='Valor Normalizado')

fig_radar = px.line_polar(df_radar_melted, r='Valor Normalizado', theta='Indicador', color='perfil',
                          line_close=True, title='<b>Personalidade de Cada Perfil Municipal</b>',
                          template='plotly_dark',
                          category_orders={'perfil': nomes_dos_perfis})
fig_radar.update_traces(fill='toself')
fig_radar.show()
# --- BOXPLOTS COM NOMES E ORDENADOS ---
for indicador in indicadores:
    fig_box = px.box(df_final, x='perfil', y=indicador, color='perfil',
                     title=f'<b>Distribuição de "{indicador.replace("_", " ").title()}" por Perfil</b>', 
                     category_orders={'perfil': nomes_dos_perfis}, # Força a ordem no eixo X
                     labels={'perfil': 'Perfil do Município'},
                     log_y=(indicador in ['pib_per_capita', 'densidade_demografica']))
    fig_box.show()

# --- (BÔNUS) MAPA DO BRASIL ---
print("\n>>> Gerando o Mapa do Brasil (pode levar um minuto)...")
# Link para o mapa dos municípios do Brasil em formato GeoJSON
url_mapa = "https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-100-mun.json"
gdf_mapa = gpd.read_file(url_mapa)
# Ajusta o tipo do código no mapa para ser igual ao nosso
gdf_mapa['id'] = gdf_mapa['id'].astype('int64')
# Junta os dados do nosso projeto com o mapa
mapa_final = gdf_mapa.merge(df_final, left_on='id', right_on='codigo_ibge', how='inner')
# Plota o mapa
fig_mapa = px.choropleth_mapbox(mapa_final, 
                               geojson=mapa_final.geometry, 
                               locations=mapa_final.index,
                               color="perfil",
                               mapbox_style="carto-positron",
                               zoom=3, center = {"lat": -15.78849, "lon": -47.88253},
                               opacity=0.7,
                               title="<b>Distribuição dos Perfis de Municípios no Brasil</b>",
                               category_orders={'perfil': nomes_dos_perfis},
                               labels={'perfil':'Perfil Municipal'})
fig_mapa.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig_mapa.show()

In [ ]:
# PASSO 6: RESULTADO FINAL (agora com nomes)
print("\n>>> [FASE 6] Salvando os resultados...")
print("\n--- PERFIL MÉDIO DE CADA GRUPO (VALORES REAIS) ---")
print(perfil_ordenado)
df_final.to_csv('resultado_segmentacao_final.csv', index=False)
print("\n>>> SUCESSO! Análise completa. O arquivo 'resultado_segmentacao_final.csv' foi salvo.")